# AI013 - Time Series Forecasting

Forecasting daily fire intensity (FRP) using three real datasets merged by date and month.

**Datasets:**
- `wildfire_multi_region_dataset.csv` — satellite fire detections, Australia 2024–2025 (hazard)
- `cybersecurity_attacks.csv` — global cyber threat incidents 2020–2023 (threats)
- `disaster-mapper-data-21-03-2023.xlsx` — AIDR historical Australian disasters 1851–2022 (misinfo/context)

**Steps:**
1. Load & prepare data
2. Create sequential inputs
3. Train forecasting model
4. Generate future predictions
5. Analyse trend behaviour
6. Compare predictions vs actual

In [ ]:
# !pip install pandas numpy scikit-learn torch matplotlib seaborn openpyxl

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('ggplot')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device} | torch: {torch.__version__}')

---
## Step 1 - Load & Prepare Data

Three datasets merged on date/month to give hazard + threat + historical context features.

In [ ]:
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'models', 'ai013-forecasting', 'data'))

FIRE_CSV     = os.path.join(DATA_DIR, 'wildfire_multi_region_dataset.csv')
CYBER_CSV    = os.path.join(DATA_DIR, 'cybersecurity_attacks.csv')
DISASTER_XLS = os.path.join(DATA_DIR, 'disaster-mapper-data-21-03-2023.xlsx')

# --- wildfire: filter Australia, aggregate to daily level ---
df_fire = pd.read_csv(FIRE_CSV)
df_fire = df_fire[df_fire['region'] == 'Australia'].copy()
df_fire['acq_date'] = pd.to_datetime(df_fire['acq_date'])

daily = df_fire.groupby('acq_date').agg(
    frp_mw       = ('frp_mw',       'mean'),
    temp_max_c   = ('temp_max_c',   'mean'),
    wind_max_kmh = ('wind_max_kmh', 'mean'),
    precip_mm    = ('precip_mm',    'mean'),
    humidity_pct = ('humidity_pct', 'mean'),
    brightness_k = ('brightness_k', 'mean'),
    event_count  = ('frp_mw',       'count')
).reset_index().sort_values('acq_date').reset_index(drop=True)

print(f'wildfire (Australia): {len(daily)} daily rows | {daily.acq_date.min().date()} to {daily.acq_date.max().date()}')
daily.head(3)

In [ ]:
# --- cyber: extract monthly averages from 2020-2023 data ---
df_cyber = pd.read_csv(CYBER_CSV)
df_cyber['Timestamp'] = pd.to_datetime(df_cyber['Timestamp'])
df_cyber['month']     = df_cyber['Timestamp'].dt.month
df_cyber['sev_num']   = df_cyber['Severity Level'].map({'Low':1,'Medium':2,'High':3})

monthly_cyber = df_cyber.groupby('month').agg(
    cyber_anomaly_score  = ('Anomaly Scores', 'mean'),
    cyber_attack_count   = ('Anomaly Scores', 'count'),
    cyber_severity_score = ('sev_num',        'mean')
).reset_index()

print(f'cyber: {len(df_cyber):,} records → {len(monthly_cyber)} monthly patterns')
monthly_cyber.round(2)

In [ ]:
# --- disaster mapper: historical Australian natural disasters (AIDR) ---
df_dis = pd.read_excel(DISASTER_XLS)
df_dis = df_dis.dropna(subset=['Start Date'])
df_dis['Start Date'] = pd.to_datetime(df_dis['Start Date'], dayfirst=True, errors='coerce')
df_dis = df_dis.dropna(subset=['Start Date'])

natural_cats = ['Bushfire','Flood','Storm','Cyclone','Heatwave','Earthquake','Tsunami','Drought']
df_dis = df_dis[df_dis['Category'].isin(natural_cats)].copy()
df_dis['Fatalities'] = pd.to_numeric(df_dis['Fatalities'], errors='coerce').fillna(0)
df_dis['month'] = df_dis['Start Date'].dt.month

monthly_hist = df_dis.groupby('month').agg(
    hist_disaster_count  = ('Category',    'count'),
    hist_fatality_rate   = ('Fatalities',  'median')  # median avoids extreme outlier skew
).reset_index()

print(f'disaster mapper: {len(df_dis)} natural disaster events (1851-2022) → monthly patterns')
monthly_hist.round(2)

In [ ]:
# merge all three datasets on month
daily['month'] = daily['acq_date'].dt.month

df = (daily
      .merge(monthly_cyber, on='month', how='left')
      .merge(monthly_hist,  on='month', how='left')
      .drop('month', axis=1)
      .sort_values('acq_date')
      .reset_index(drop=True))

print(f'final dataset: {df.shape} | nulls: {df.isnull().sum().sum()}')
df.head()

In [ ]:
FEATURE_COLS = [
    # hazard (wildfire)
    'temp_max_c', 'wind_max_kmh', 'precip_mm', 'humidity_pct', 'brightness_k', 'event_count',
    # threats (cyber)
    'cyber_anomaly_score', 'cyber_attack_count', 'cyber_severity_score',
    # historical context (disaster mapper)
    'hist_disaster_count', 'hist_fatality_rate'
]
TARGET = 'frp_mw'

print(f'features: {len(FEATURE_COLS)} | target: {TARGET}')
print()
print(df[FEATURE_COLS + [TARGET]].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['acq_date'], df[TARGET], color='#c0392b', linewidth=1.5, marker='o', markersize=3)
ax.set_title('Daily Fire Radiative Power — Australia 2024–2025', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('FRP (MW)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('fire_intensity_over_time.png', dpi=150)
plt.show()

In [ ]:
corr = df[FEATURE_COLS + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix — All 3 Datasets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
frp_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colours = ['#e74c3c' if v > 0 else '#2980b9' for v in frp_corr]
frp_corr.plot(kind='barh', ax=ax, color=colours, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with FRP Target', fontsize=12, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150)
plt.show()

---
## Step 2 - Create Sequential Inputs

Sliding window of 5 past days → predict next day FRP.

In [ ]:
feat_scaler   = MinMaxScaler()
target_scaler = MinMaxScaler()

X_scaled = feat_scaler.fit_transform(df[FEATURE_COLS].values)
y_scaled = target_scaler.fit_transform(df[[TARGET]].values)

WINDOW = 5

def make_sequences(features, target, window):
    X_out, y_out = [], []
    for i in range(len(features) - window):
        X_out.append(features[i : i + window])
        y_out.append(target[i + window])
    return np.array(X_out), np.array(y_out)

X_seq, y_seq = make_sequences(X_scaled, y_scaled, WINDOW)
print(f'sequences → X: {X_seq.shape}  y: {y_seq.shape}')

In [ ]:
TRAIN_SPLIT = 0.80
split = int(len(X_seq) * TRAIN_SPLIT)

X_train_t = torch.tensor(X_seq[:split], dtype=torch.float32)
y_train_t = torch.tensor(y_seq[:split], dtype=torch.float32)
X_test_t  = torch.tensor(X_seq[split:], dtype=torch.float32)
y_test_t  = torch.tensor(y_seq[split:], dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=False)
print(f'train: {len(X_train_t)} | test: {len(X_test_t)}')

---
## Step 3 - Train Forecasting Model

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=32, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))


INPUT_SIZE  = len(FEATURE_COLS)
HIDDEN_SIZE = 32
EPOCHS      = 150
LR          = 0.001

model     = LSTMForecaster(INPUT_SIZE, HIDDEN_SIZE).to(device)
loss_fn   = nn.MSELoss()
optimiser = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=20, factor=0.5)

print(model)
print(f'trainable params: {sum(p.numel() for p in model.parameters()):,}')

train_losses, val_losses = [], []
best_val, best_weights   = float('inf'), None

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimiser.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        batch_losses.append(loss.item())

    avg_train = np.mean(batch_losses)
    train_losses.append(avg_train)

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_test_t.to(device)), y_test_t.to(device)).item()
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val     = val_loss
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 50 == 0:
        print(f'epoch {epoch:3d}  train: {avg_train:.5f}  val: {val_loss:.5f}')

model.load_state_dict(best_weights)
print(f'\nbest val loss: {best_val:.5f}')

In [ ]:
baseline_scaled = np.array([
    y_scaled[split + i - WINDOW : split + i].mean()
    for i in range(len(X_test_t))
]).reshape(-1, 1)
baseline_preds = target_scaler.inverse_transform(baseline_scaled)
print('baseline ready')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train', color='#2980b9', linewidth=1.5)
ax.plot(val_losses,   label='Val',   color='#e74c3c', linewidth=1.5)
ax.set_title('Training Loss (MSE)', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()

---
## Step 4 - Generate Future Predictions

In [ ]:
model.eval()
with torch.no_grad():
    test_pred_scaled = model(X_test_t.to(device)).cpu().numpy()

test_pred_actual = target_scaler.inverse_transform(test_pred_scaled)
y_test_actual    = target_scaler.inverse_transform(y_test_t.numpy())

print('predicted vs actual (test set):')
for i in range(len(test_pred_actual)):
    print(f'  pred {test_pred_actual[i][0]:8.2f} MW  |  actual {y_test_actual[i][0]:8.2f} MW')

In [ ]:
FUTURE_STEPS = 14

def forecast_ahead(model, seed_window, steps, device):
    model.eval()
    preds, window = [], seed_window.copy()
    with torch.no_grad():
        for _ in range(steps):
            inp  = torch.tensor(window[np.newaxis], dtype=torch.float32).to(device)
            p    = model(inp).cpu().numpy()[0][0]
            preds.append(p)
            next_row    = window[-1].copy()
            next_row[0] = p
            window      = np.vstack([window[1:], next_row])
    return np.array(preds)

future_scaled = forecast_ahead(model, X_scaled[-WINDOW:], FUTURE_STEPS, device)
future_actual = target_scaler.inverse_transform(future_scaled.reshape(-1, 1))
future_dates  = pd.date_range(start=df['acq_date'].max(), periods=FUTURE_STEPS + 1, freq='D')[1:]

future_df = pd.DataFrame({'date': future_dates, 'predicted_frp_mw': future_actual.flatten()})
print(future_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['acq_date'], df[TARGET], color='#2c3e50', linewidth=1.3, alpha=0.7, label='Historical')
ax.plot(future_df['date'], future_df['predicted_frp_mw'],
        color='#e74c3c', linewidth=2, linestyle='--', marker='s', markersize=6, label='Forecast')
ax.axvspan(future_df['date'].iloc[0], future_df['date'].iloc[-1],
           alpha=0.08, color='red', label='Forecast Window')
ax.set_title('Fire Intensity — Historical + 14-Day Forecast', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('FRP (MW)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('future_forecast.png', dpi=150)
plt.show()

In [ ]:
os.makedirs('outputs', exist_ok=True)

test_start = split + WINDOW
test_dates = df['acq_date'].iloc[test_start : test_start + len(y_test_actual)].values

results_df = pd.DataFrame({
    'date'        : test_dates,
    'actual_frp'  : y_test_actual.flatten(),
    'pred_frp'    : test_pred_actual.flatten(),
    'baseline_frp': baseline_preds.flatten()
})
results_df.to_csv('outputs/ai013_test_predictions.csv', index=False)
print('saved → outputs/ai013_test_predictions.csv')
print(results_df.round(2).to_string(index=False))

---
## Step 5 - Analyse Trend Behaviour

In [ ]:
df['rolling_mean'] = df[TARGET].rolling(window=7, min_periods=1).mean()
df['rolling_std']  = df[TARGET].rolling(window=7, min_periods=1).std().fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['acq_date'], df[TARGET], alpha=0.3, color='#7f8c8d', linewidth=1, label='Raw FRP')
ax.plot(df['acq_date'], df['rolling_mean'], color='#2980b9', linewidth=2.5, label='7-Day Rolling Mean')
ax.fill_between(df['acq_date'],
                df['rolling_mean'] - df['rolling_std'],
                df['rolling_mean'] + df['rolling_std'],
                alpha=0.15, color='#2980b9', label='±1 Std Dev')
ax.set_title('Fire Intensity Trend — Rolling Average', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('FRP (MW)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('trend_rolling.png', dpi=150)
plt.show()

In [ ]:
df['month'] = df['acq_date'].dt.month
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_avg  = df.groupby('month')[TARGET].mean()

fig, ax = plt.subplots(figsize=(10, 4))
monthly_avg.plot(kind='bar', ax=ax, color='#8e44ad', edgecolor='black')
ax.set_title('Average Fire Intensity by Month', fontsize=12, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Avg FRP (MW)')
ax.set_xticklabels([month_labels[m - 1] for m in monthly_avg.index], rotation=0)
plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150)
plt.show()

In [ ]:
# compare all 3 data sources by month
monthly_cyber_avg = df.groupby('month')['cyber_anomaly_score'].mean()
monthly_hist_avg  = df.groupby('month')['hist_disaster_count'].mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].bar(monthly_avg.index, monthly_avg.values, color='#e74c3c', edgecolor='black')
axes[0].set_title('Avg Fire Intensity (FRP) by Month', fontweight='bold')
axes[0].set_xlabel('Month'); axes[0].set_ylabel('FRP (MW)')
axes[0].set_xticks(monthly_avg.index)
axes[0].set_xticklabels([month_labels[m-1] for m in monthly_avg.index], rotation=45)

axes[1].bar(monthly_cyber_avg.index, monthly_cyber_avg.values, color='#2980b9', edgecolor='black')
axes[1].set_title('Avg Cyber Anomaly Score by Month', fontweight='bold')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('Anomaly Score')
axes[1].set_xticks(monthly_cyber_avg.index)
axes[1].set_xticklabels([month_labels[m-1] for m in monthly_cyber_avg.index], rotation=45)

axes[2].bar(monthly_hist_avg.index, monthly_hist_avg.values, color='#27ae60', edgecolor='black')
axes[2].set_title('Historical Disaster Count by Month (AIDR)', fontweight='bold')
axes[2].set_xlabel('Month'); axes[2].set_ylabel('Disaster Count')
axes[2].set_xticks(monthly_hist_avg.index)
axes[2].set_xticklabels([month_labels[m-1] for m in monthly_hist_avg.index], rotation=45)

plt.suptitle('Hazard | Cyber Threat | Historical Disasters — Monthly Patterns', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('all_datasets_monthly.png', dpi=150)
plt.show()

---
## Step 6 - Compare Predictions vs Actual

In [ ]:
mae_lstm  = mean_absolute_error(y_test_actual, test_pred_actual)
rmse_lstm = np.sqrt(mean_squared_error(y_test_actual, test_pred_actual))
r2_lstm   = r2_score(y_test_actual, test_pred_actual)

mae_base  = mean_absolute_error(y_test_actual, baseline_preds)
rmse_base = np.sqrt(mean_squared_error(y_test_actual, baseline_preds))
r2_base   = r2_score(y_test_actual, baseline_preds)

print(f'{"":<20} {"LSTM":>10} {"Baseline":>10}')
print('-' * 42)
print(f'{"MAE":<20} {mae_lstm:>10.4f} {mae_base:>10.4f}')
print(f'{"RMSE":<20} {rmse_lstm:>10.4f} {rmse_base:>10.4f}')
print(f'{"R2":<20} {r2_lstm:>10.4f} {r2_base:>10.4f}')
print()
print(f'lower MAE: {"LSTM" if mae_lstm < mae_base else "Baseline"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(test_dates, y_test_actual,    color='#2c3e50', linewidth=2, marker='o', markersize=5, label='Actual')
axes[0].plot(test_dates, test_pred_actual, color='#e74c3c', linewidth=1.8, linestyle='--', marker='s', markersize=4, label='LSTM')
axes[0].plot(test_dates, baseline_preds,   color='#27ae60', linewidth=1.5, linestyle=':', marker='^', markersize=4, label='Baseline')
axes[0].set_title('Predicted vs Actual vs Baseline', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('FRP (MW)')
axes[0].legend()

lo = min(y_test_actual.min(), test_pred_actual.min()) - 5
hi = max(y_test_actual.max(), test_pred_actual.max()) + 5
axes[1].scatter(y_test_actual, test_pred_actual, color='#e74c3c', alpha=0.8, edgecolors='black', s=80, label='LSTM')
axes[1].scatter(y_test_actual, baseline_preds,   color='#27ae60', alpha=0.6, edgecolors='black', s=60, marker='^', label='Baseline')
axes[1].plot([lo, hi], [lo, hi], 'k--', linewidth=1.5, label='Perfect')
axes[1].set_title(f'Scatter — LSTM R²={r2_lstm:.3f}', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Actual FRP (MW)'); axes[1].set_ylabel('Predicted FRP (MW)')
axes[1].legend()

plt.tight_layout()
plt.savefig('predictions_vs_actual.png', dpi=150)
plt.show()

In [ ]:
residuals = y_test_actual.flatten() - test_pred_actual.flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(len(residuals)), residuals,
            color=['#27ae60' if r >= 0 else '#e74c3c' for r in residuals])
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Residuals per Test Sample', fontweight='bold')
axes[0].set_xlabel('Sample'); axes[0].set_ylabel('Actual - Predicted')

axes[1].hist(residuals, bins=8, color='#8e44ad', edgecolor='black', alpha=0.8)
axes[1].axvline(0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('residuals.png', dpi=150)
plt.show()
print(f'mean residual: {residuals.mean():.2f}')

In [ ]:
x = np.arange(2)
w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, [mae_lstm, rmse_lstm], w, label='LSTM',     color='#e74c3c', edgecolor='black')
ax.bar(x + w/2, [mae_base, rmse_base], w, label='Baseline', color='#27ae60', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(['MAE', 'RMSE'])
ax.set_title('LSTM vs Baseline — Error Comparison', fontsize=12, fontweight='bold')
ax.set_ylabel('Error')
ax.legend()
plt.tight_layout()
plt.savefig('model_vs_baseline.png', dpi=150)
plt.show()